In [1]:
import random
import pandas as pd

PII_TYPES = ["NAME", "EMAIL", "ADDRESS", "PHONE"]

def make_fake_record():
    """간단한 PII가 들어간 가짜 문장 하나 생성 + 정답 라벨 세트 반환"""
    names = ["Alice Byrne", "Kim Minji", "Liam O'Connor", "Nahyung Choi"]
    cities = ["Dublin", "Cork", "Seoul", "Busan"]
    streets = ["High Street", "Main Street", "River Road", "Park Avenue"]

    name = random.choice(names)
    city = random.choice(cities)
    street = random.choice(streets)
    email = name.lower().replace(" ", ".") + "@example.com"
    phone = f"+353-87-{random.randint(1000000, 9999999)}"
    address = f"{random.randint(1, 99)} {street}, {city}"

    # 여러 유형 섞어서 문장 만들기
    templates = [
        f"The patient {name} lives at {address}. Contact: {email}, phone {phone}.",
        f"{name} registered with email {email}. Phone number is {phone}.",
        f"Delivery address: {address}. Recipient: {name}.",
        f"For inquiries, contact {name} via {email}.",
        f"Please call {phone} to reach {name} in {city}.",
    ]
    text = random.choice(templates)

    labels = set()
    if name in text:
        labels.add("NAME")
    if email in text:
        labels.add("EMAIL")
    if address.split(",")[0] in text or address in text:
        labels.add("ADDRESS")
    if phone in text:
        labels.add("PHONE")

    return text, labels

def build_synthetic_dataset(n=50, seed=42):
    random.seed(seed)
    records = []
    for _ in range(n):
        text, labels = make_fake_record()
        records.append({
            "text": text,
            "labels": list(labels)  # 예: ["NAME","EMAIL"]
        })
    return pd.DataFrame(records)

df = build_synthetic_dataset(n=30)
df.head()


,text,labels
0,Alice Byrne registered with email alice.byrne@...,"[NAME, PHONE, EMAIL]"
1,The patient Alice Byrne lives at 4 Park Avenue...,"[NAME, PHONE, ADDRESS, EMAIL]"
2,Please call +353-87-4335942 to reach Kim Minji...,"[NAME, PHONE]"
3,Nahyung Choi registered with email nahyung.cho...,"[NAME, PHONE, EMAIL]"
4,"Delivery address: 28 River Road, Seoul. Recipi...","[NAME, ADDRESS]"


In [2]:
import re

EMAIL_REGEX = re.compile(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}")
PHONE_REGEX = re.compile(r"\+\d{2,3}-\d{2,3}-\d{6,8}")

def detect_pii(text: str) -> set:
    """
    현재는 단순 regex 기반 baseline.
    나중에 여기를 네 LLM 파이프라인으로 교체하면 됨.
    반환: 예측된 PII 유형 집합 (예: {"NAME","EMAIL"})
    """
    pred = set()

    # 이메일 패턴
    if EMAIL_REGEX.search(text):
        pred.add("EMAIL")

    # 전화번호 패턴
    if PHONE_REGEX.search(text):
        pred.add("PHONE")

    # 이름/주소는 아주 단순한 heuristic (baseline이라 대충)
    # 실제로는 LLM이나 NER 모델을 사용할 것
    if any(tok in text for tok in ["Alice", "Kim", "Liam", "Nahyung"]):
        pred.add("NAME")
    if any(tok in text for tok in ["Street", "Road", "Avenue", "Delivery address"]):
        pred.add("ADDRESS")

    return pred


In [3]:
from sklearn.metrics import precision_score, recall_score, f1_score

def evaluate_dataset(df: pd.DataFrame, pii_types=PII_TYPES):
    """
    각 PII 유형(NAME/EMAIL/ADDRESS/PHONE)에 대해
    precision / recall / f1을 계산해서 DataFrame으로 반환
    """
    rows = []

    for pii in pii_types:
        y_true = []
        y_pred = []

        for _, row in df.iterrows():
            gold_labels = set(row["labels"])      # 정답 레이블 집합
            pred_labels = detect_pii(row["text"]) # 예측 레이블 집합

            y_true.append(1 if pii in gold_labels else 0)
            y_pred.append(1 if pii in pred_labels else 0)

        p = precision_score(y_true, y_pred, zero_division=0)
        r = recall_score(y_true, y_pred, zero_division=0)
        f = f1_score(y_true, y_pred, zero_division=0)

        rows.append({
            "pii_type": pii,
            "precision": p,
            "recall": r,
            "f1": f,
            "support": sum(y_true),   # 해당 PII가 실제로 등장한 문서 수
        })

    return pd.DataFrame(rows)

metrics_df = evaluate_dataset(df)
metrics_df


,pii_type,precision,recall,f1,support
0,NAME,1.0,1.0,1.0,30
1,EMAIL,1.0,1.0,1.0,15
2,ADDRESS,1.0,1.0,1.0,17
3,PHONE,1.0,1.0,1.0,15
